
# Hybrid SNN (threshold-reset) with Adjoint/Backprop + Surrogate Gradients

This notebook implements training of Spikin Neural Networks (SNN) via backpropagation/adjoint, using **surrogate gradients** to handle the non-differentiability at spike/reset events.

**What’s included**
- LIF hybrid neurons with instantaneous reset to 0 when crossing a threshold.
- Hidden layers receive **spike trains** from the previous layer and convert them into synaptic currents via **Gaussian kernels** (as in the paper’s Section 2 formulae).
- Output layer LIF neurons **without reset**.
- Adjoint/backpropagation implemented via PyTorch autograd; the surrogate derivative is used for the Heaviside/threshold operation, which makes the backward pass well-defined (this is the discrete-time adjoint of the time-march).
- Minimal synthetic example to show end-to-end training.

In [1]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.nn.utils as nn_utils
from torch.optim import Adam, AdamW
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
import numpy as np

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


## Surrogate spike function

We use a Heaviside-like function in the forward pass and a smooth **surrogate derivative** in the backward pass. 


In [2]:
def alpha_schedule(epoch, total_epochs, a0=3.0, a1=15.0):
    
    return a0 * (a1 / a0) ** (epoch / max(1, total_epochs - 1))

class SurrogateHeaviside(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, alpha):
        # Hard threshold in forward (hybrid reset detection)
        out = (x > 0.0).to(x.dtype)
        # Save x for surrogate derivative
        ctx.save_for_backward(x, alpha)
        return out

    @staticmethod
    def backward(ctx, grad_output):
        x, alpha = ctx.saved_tensors        
        z = (alpha * x) / 2.0
        sech2 = 1.0 / (torch.cosh(z)**2 + 1e-12)
        grad = (alpha / 4.0) * sech2
        return grad_output * grad, None

def spike_fn(x, alpha):
    if not torch.is_tensor(alpha):
        alpha = torch.tensor(alpha, device=x.device, dtype=x.dtype)
    return SurrogateHeaviside.apply(x, alpha)


## Gaussian synaptic kernel

Hidden and output layers receive currents given by sums of Gaussians centered at **presynaptic spike times**. We implement a causal discrete-time approximation by splatting presynaptic spikes 
into a time-series via a Gaussian kernel.


In [3]:
def gaussian_kernel_1d(times, centers, mu):
    
    t = times.view(*(1,)* (centers.dim()) + (times.numel(),))  
    c = centers.unsqueeze(-1)                                  
    diff2 = (t - c)**2    
    g = torch.exp(-0.5 * diff2 / (mu**2 + 1e-12))
    return g.sum(dim=-2)  

## LIF Hybrid Cells

- **Hidden layers**: LIF with hard reset. When the potential crosses the threshold we emit a spike and reset to 0.
- **Output layer**: LIF **without** reset.

We discretize time with step and use explicit Euler. The discrete-time adjoint is simply PyTorch's autograd given our computational graph, with surrogate gradients making the spike/reset differentiable for learning.

In [4]:
class LIFHybridHidden(nn.Module):
    def __init__(self, P, tau, theta, mu, dt, T_steps):
        super().__init__()
        self.P = P
        self.tau = torch.tensor(tau, dtype=torch.float32)
        self.theta = torch.tensor(theta, dtype=torch.float32)
        self.mu = torch.tensor(mu, dtype=torch.float32)
        self.dt = torch.tensor(dt, dtype=torch.float32)
        self.T_steps = T_steps
        
        self.omega = nn.Parameter(torch.randn(P) * 0.5)
        self.register_buffer("alpha", torch.tensor(10.0, dtype=torch.float32))    

    def forward(self, presyn_spikes):
            
        B, P_prev, T = presyn_spikes.shape
        device = presyn_spikes.device
        dtype = presyn_spikes.dtype
    
        # ----- differentiable Gaussian synaptic drive -----
        # Sum spikes across all presynaptic neurons, then convolve in time.        
        def _gauss_kernel(mu):
            mu = float(self.mu)  
            radius = max(1, int(4.0 * mu))
            t = torch.arange(-radius, radius + 1, device=device, dtype=dtype)
            g = torch.exp(-0.5 * (t / mu) ** 2)
            g = g / (g.sum() + 1e-12)            
            w = g.view(1, 1, -1)                 
            pad = radius
            return w, pad
    
        w, pad = _gauss_kernel(self.mu)
    
        s_prev = presyn_spikes.sum(dim=1, keepdim=True)              
        J = F.conv1d(s_prev, w, padding=pad)                         
    
        J = J.expand(B, self.P, T)                                   
    
        # ----- evolve LIF with hard reset (hybrid) -----
        v = torch.zeros(B, self.P, device=device, dtype=dtype)
        spikes = torch.zeros(B, self.P, T, device=device, dtype=dtype)
        omega = self.omega.view(1, self.P)
    
        for t in range(T):
            dv = (-(v) + omega * J[:, :, t]) * (self.dt / self.tau)
            v = v + dv
    
            over = v - self.theta
            s = spike_fn(over, alpha=self.alpha)       
            spikes[:, :, t] = s
    
            v = v * (1.0 - s)                    
    
        return spikes

class LIFOutputNoReset(nn.Module):
    def __init__(self, P, tau_u, w, mu, dt, T_steps):
        super().__init__()
        self.P = P
        self.tau_u = torch.tensor(tau_u, dtype=torch.float32)
        self.mu = torch.tensor(mu, dtype=torch.float32)
        self.dt = torch.tensor(dt, dtype=torch.float32)        
        self.w = nn.Parameter(torch.tensor(float(w)))
        self.register_buffer("alpha", torch.tensor(10.0, dtype=torch.float32))
    
    def forward(self, last_layer_spikes):
            
        B, P, T = last_layer_spikes.shape
        device = last_layer_spikes.device
        dtype = last_layer_spikes.dtype
    
        # ----- differentiable Gaussian smoothing per neuron -----
        def _gauss_kernel(mu):
            mu = float(self.mu)
            radius = max(1, int(4.0 * mu))
            t = torch.arange(-radius, radius + 1, device=device, dtype=dtype)
            g = torch.exp(-0.5 * (t / mu) ** 2)
            g = g / (g.sum() + 1e-12)                             
            w = g.view(1, 1, -1).expand(P, 1, -1).contiguous() 
            pad = radius
            return w, pad
    
        w, pad = _gauss_kernel(self.mu)
    
        Phi = F.conv1d(last_layer_spikes, w, padding=pad, groups=P)   
        
        # ----- LIF without reset -----
        u = torch.zeros(B, P, device=device, dtype=dtype)
        for t in range(T):
            du = (-(u) + self.w * Phi[:, :, t]) * (self.dt / self.tau_u)
            u = u + du
    
        return u

## Full SNN model

- Input "layer": one neuron converting input data to a spike train using a simple LIF with reset (hybrid). 
- Hidden layers: `LIFHybridHidden`
- Output layer: `LIFOutputNoReset`
- Readout: Cybenko-style ansatz.

In [5]:
class InputLIFHybrid(nn.Module):
    def __init__(self, d, tau_v, theta_v, dt, T_steps):
        super().__init__()
        self.d = d
        self.tau_v = torch.tensor(tau_v, dtype=torch.float32)
        self.theta_v = torch.tensor(theta_v, dtype=torch.float32)
        self.dt = torch.tensor(dt, dtype=torch.float32)        
        self.a = nn.Parameter(torch.randn(d) * 0.5)
        self.register_buffer("alpha", torch.tensor(10.0, dtype=torch.float32))

    def forward(self, x, T_steps):
        """
        x: (batch, d)
        Returns input spikes: (batch, 1, T)
        Drive: <a, x> constant over time, LIF with reset.
        """
        B, d = x.shape
        drive = torch.matmul(x, self.a)  # (B,)
        v = torch.zeros(B, device=x.device)
        spikes = torch.zeros(B, 1, T_steps, device=x.device)
        for t in range(T_steps):
            dv = (-(v) + drive) * (self.dt / self.tau_v)
            v = v + dv
            over = v - self.theta_v            
            s = spike_fn(over, self.alpha)
            spikes[:, 0, t] = s
            v = v * (1.0 - s)  
        return spikes

class ReadoutCybenko(nn.Module):
    def __init__(self, P, theta_u):
        super().__init__()
        self.theta_u = torch.tensor(theta_u, dtype=torch.float32)
        self.nu = nn.Parameter(torch.randn(P) * 0.5)

    def forward(self, uT):        
        z = torch.sigmoid(uT - self.theta_u)  
        y = torch.sum(self.nu.unsqueeze(0) * z, dim=1, keepdim=True)
        return y, z

class HybridSNN(nn.Module):
    def __init__(self, d, P, L, params):
        super().__init__()
        self.d, self.P, self.L = d, P, L
        self.dt = params['dt']
        self.T_steps = params['T_steps']

        # Layers
        self.input_layer = InputLIFHybrid(d, params['tau_v'], params['theta_v'], 
                                          self.dt, self.T_steps)
        self.hidden = nn.ModuleList()
        for _ in range(L):
            self.hidden.append(LIFHybridHidden(P, params['tau_h'], params['theta_h'], 
                                               params['mu'], self.dt, self.T_steps))
        self.output_layer = LIFOutputNoReset(P, params['tau_u'], params['w_init'], 
                                             params['mu'], self.dt, self.T_steps)
        self.readout = ReadoutCybenko(P, params['theta_u'])

    def forward(self, x):
        s = self.input_layer(x, self.T_steps)        
        for layer in self.hidden:
            s = layer(s)                             
        uT = self.output_layer(s)                    
        y, z = self.readout(uT)                      
        
        return y, uT, z

## Taining 

We fit a simple binary function on to verify that gradients flow through spikes/resets thanks to the surrogate derivative.

In [6]:
dataset = '/moons/'

X, y = make_moons(n_samples=250, noise=0.0, random_state=0)
X = (X - X.mean(0)) / X.std(0)
d = X.shape[1]

Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)
Xtr, Xval, ytr, yval = train_test_split(Xtr, ytr, test_size=0.1, random_state=0, stratify=ytr)

x_train = torch.tensor(Xtr, dtype=torch.float32, device=device)
x_val = torch.tensor(Xval, dtype=torch.float32, device=device)
x_test = torch.tensor(Xte, dtype=torch.float32, device=device)
y_train = torch.tensor(ytr, dtype=torch.float32, device=device)[:, None]
y_val = torch.tensor(yval, dtype=torch.float32, device=device)[:, None]
y_test = torch.tensor(yte, dtype=torch.float32, device=device)[:, None]

In [7]:
params = dict(
    dt=1.0, T_steps=60,
    tau_v=8.0, theta_v=0.8,
    tau_h=6.0, theta_h=0.25,
    tau_u=10.0, theta_u=0.3,
    mu=0.2,
    w_init=0.5
)

model = HybridSNN(d=d, P=8, L=1, params=params).to(device)
opt = AdamW(model.parameters(), lr=1e-1, weight_decay=1e-4)

def _to_np(x):
    return x.detach().float().cpu().numpy()

def _bin(p):
    return (p > 0.5).float()

def compute_metrics(y_true_bin, p_prob):
    
    y_true_np = _to_np(y_true_bin.view(-1))
    y_pred_np = _to_np(_bin(p_prob).view(-1))
    return {
        "acc": accuracy_score(y_true_np, y_pred_np),
        "f1": f1_score(y_true_np, y_pred_np, average="binary", zero_division=0),
        "prec": precision_score(y_true_np, y_pred_np, average="binary", zero_division=0),
        "rec": recall_score(y_true_np, y_pred_np, average="binary", zero_division=0),
    }

def bce_logits_from_readout(readout_y):
    return torch.sigmoid(readout_y)

def lr_at_epoch(e):
    if e < warmup_epochs:        
        return base_lr
    
    t = (e - warmup_epochs) / max(1, (num_epochs - warmup_epochs))
    return min_lr + 0.5*(base_lr - min_lr)*(1 + math.cos(math.pi * t))

num_epochs = 15
base_lr = 1
min_lr  = 1e-4
warmup_epochs = 60

train_losses, train_acc, train_f1, train_prec, train_rec = [], [], [], [], []
val_losses, val_acc, val_f1, val_prec, val_rec = [], [], [], [], []

for epoch in range(num_epochs):    
    # ---- Train step ----
    for g in opt.param_groups:
        g["lr"] = lr_at_epoch(epoch)
    model.train()
    opt.zero_grad()
    cur_alpha = alpha_schedule(epoch, num_epochs, a0=3.0, a1=15.0)
    for m in model.modules():
        if hasattr(m, "alpha"):
            m.alpha.fill_(float(cur_alpha))
    y_pred_train, _, _ = model(x_train)        # (B,1)
    p_train = bce_logits_from_readout(y_pred_train)    
    train_metrics = compute_metrics(y_train, p_train)

    train_acc.append(train_metrics['acc'])
    train_f1.append(train_metrics['f1'])
    train_prec.append(train_metrics['prec'])
    train_rec.append(train_metrics['rec'])

    loss_fn = torch.nn.BCEWithLogitsLoss()
    loss = F.binary_cross_entropy(p_train, y_train)    
    train_losses.append(loss.item())

    loss.backward()

    opt.step()

    # ---- Validation ----
    model.eval()
    with torch.no_grad():
        y_pred_val, _, _ = model(x_val)
        p_val = bce_logits_from_readout(y_pred_val)
        val_loss = F.binary_cross_entropy(p_val, y_val)
        val_losses.append(val_loss.item())
        val_metrics = compute_metrics(y_val, p_val)

    val_acc.append(val_metrics['acc'])
    val_f1.append(val_metrics['f1'])
    val_prec.append(val_metrics['prec'])
    val_rec.append(val_metrics['rec'])

    if (epoch + 1) % 10 == 0 or epoch == 0:
        print(
            f"Epoch {epoch+1:02d} | "
            f"loss {loss.item():.4f} | "
            f"lr {lr_at_epoch(epoch):.4f} | "
            f"val acc {val_metrics['acc']:.3f} f1 {val_metrics['f1']:.3f} "
            f"prec {val_metrics['prec']:.3f} rec {val_metrics['rec']:.3f}"
        )

Epoch 01 | loss 0.7352 | lr 1.0000 | val acc 0.474 f1 0.000 prec 0.000 rec 0.000


KeyboardInterrupt: 

In [ ]:
# ===== Final test evaluation =====
from matplotlib import pyplot as plt

model.eval()
with torch.no_grad():
    y_pred_test, _, _ = model(x_test)
    p_test = bce_logits_from_readout(y_pred_test)
    test_loss = F.binary_cross_entropy(p_test, y_test)

test_metrics = compute_metrics(y_test, p_test)
y_true_np = _to_np(y_test.view(-1))
p_test_np = _to_np(p_test.view(-1))

# Compute ROC-AUC
auc_score = roc_auc_score(y_true_np, p_test_np)
fpr, tpr, _ = roc_curve(y_true_np, p_test_np)

print("\n=== Final Test Evaluation ===")
for k, v in test_metrics.items():
    print(f"{k.upper():>5}: {v:.3f}")
print(f"AUC : {auc_score:.3f}")

# Plot ROC curve
plt.figure(figsize=(6, 5))
plt.plot(fpr, tpr, label=f"ROC curve (AUC = {auc_score:.3f})")
plt.plot([0, 1], [0, 1], 'k--', alpha=0.6)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Test ROC Curve")
plt.legend()
plt.grid(alpha=0.3)
plt.show()